In [ ]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 106.6 MB/s eta 0:00:00


#test run

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("1. Initializing Toy Medical Library...")
toy_documents = [
    "Aspirin is a salicylate drug often used as an analgesic to relieve minor aches and pains, as an antipyretic to reduce fever, and as an anti-inflammatory medication.",
    "The mitochondria are double-membrane-bound organelles found in most eukaryotic organisms, heavily involved in programmed cell death.",
    "Strabismus is a condition in which the eyes do not properly align with each other when looking at an object, potentially leading to amblyopia."
]

print("2. Loading Embedding Model (CPU Friendly)...")
# This will download quickly if not already cached
embedder = SentenceTransformer('all-MiniLM-L6-v2')
dim = embedder.get_sentence_embedding_dimension()

print("3. Embedding Documents...")
# Encode the 3 documents into vectors
doc_embeddings = embedder.encode(toy_documents)
doc_embeddings = np.array(doc_embeddings).astype("float32")

# Normalize for Cosine Similarity (Inner Product)
faiss.normalize_L2(doc_embeddings)

print("4. Building FAISS Index...")
vector_db = faiss.IndexFlatIP(dim)
vector_db.add(doc_embeddings)
print(f"   Success! Index contains {vector_db.ntotal} vectors.")

print("\n--- 5. TESTING RETRIEVAL LOGIC ---")
# Let's ask a question that relates to the first document but doesn't use the exact words
test_query = "What common medication is used to lower a patient's temperature and reduce swelling?"
print(f"User Query: '{test_query}'")

# Embed the query
query_embedding = embedder.encode([test_query])
query_embedding = np.array(query_embedding).astype("float32")
faiss.normalize_L2(query_embedding)

# Search the FAISS database for the Top 1 most similar document
k = 1
distances, indices = vector_db.search(query_embedding, k)

# Extract results
retrieved_doc_index = indices[0][0]
similarity_score = distances[0][0]

print("\n✅ RETRIEVAL RESULTS:")
print(f"Top Match Score: {similarity_score:.4f} (1.0 is a perfect match)")
print(f"Retrieved Document: '{toy_documents[retrieved_doc_index]}'")

if retrieved_doc_index == 0:
    print("\n🟢 LOGIC TEST PASSED: The system successfully retrieved the Aspirin document based on semantic meaning!")
else:
    print("\n🚨 LOGIC TEST FAILED: The system retrieved the wrong document.")

1. Initializing Toy Medical Library...
2. Loading Embedding Model (CPU Friendly)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

3. Embedding Documents...
4. Building FAISS Index...
   Success! Index contains 3 vectors.

--- 5. TESTING RETRIEVAL LOGIC ---
User Query: 'What common medication is used to lower a patient's temperature and reduce swelling?'

✅ RETRIEVAL RESULTS:
Top Match Score: 0.3727 (1.0 is a perfect match)
Retrieved Document: 'Aspirin is a salicylate drug often used as an analgesic to relieve minor aches and pains, as an antipyretic to reduce fever, and as an anti-inflammatory medication.'

🟢 LOGIC TEST PASSED: The system successfully retrieved the Aspirin document based on semantic meaning!


#bbgurllll

In [ ]:
import pandas as pd
import numpy as np
import faiss
import pickle
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

from datasets import concatenate_datasets

print("1. Loading ALL PubMedQA Subsets...")
# Download all three subsets
ds_labeled = load_dataset("pubmed_qa", "pqa_labeled", split="train")
ds_artificial = load_dataset("pubmed_qa", "pqa_artificial", split="train")
ds_unlabeled = load_dataset("pubmed_qa", "pqa_unlabeled", split="train")

print("2. Extracting and Deduplicating Medical Abstracts...")
document_library = {}

# We create a single list of all three datasets to loop through
all_datasets = [ds_labeled, ds_artificial, ds_unlabeled]

for ds in all_datasets:
    for row in tqdm(ds):
        # Extract only the abstract text, completely ignoring the Q&A
        full_abstract = " ".join(row['context']['contexts'])
        pubmed_id = row['pubid']

        # Deduplicate using the PubMed ID
        if pubmed_id not in document_library:
            document_library[pubmed_id] = full_abstract

print(f"\nExtracted {len(document_library)} unique medical documents across all subsets!")

print("\n3. Loading Embedding Model...")
# all-MiniLM-L6-v2 is perfect here: it's fast enough to embed 200k docs in minutes
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dimension = embedder.get_sentence_embedding_dimension()

print("4. Embedding Documents into Vectors (Grab a coffee, this takes 15-30 mins on GPU)...")
doc_ids = list(document_library.keys())
doc_texts = list(document_library.values())

# Encode all texts into a numpy matrix
document_embeddings = embedder.encode(doc_texts, show_progress_bar=True, batch_size=256)
document_embeddings = np.array(document_embeddings).astype("float32")

print("\n5. Building the FAISS Vector Database...")
faiss.normalize_L2(document_embeddings)
vector_db = faiss.IndexFlatIP(embedding_dimension)
vector_db.add(document_embeddings)

print(f"FAISS Index built! Total vectors stored: {vector_db.ntotal}")

print("\n6. Saving the Database to Disk...")
faiss.write_index(vector_db, "pubmed_massive_faiss.index")

# Save the mapping so we can retrieve the text later
with open("pubmed_massive_mapping.pkl", "wb") as f:
    pickle.dump({"ids": doc_ids, "texts": doc_texts}, f)

print("✅ Massive Baseline RAG Backend is ready!")

1. Loading ALL PubMedQA Subsets...


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/233M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

4. Embedding Documents into Vectors (Grab a coffee, this takes 15-30 mins on GPU)...


Batches:   0%|          | 0/1069 [00:00<?, ?it/s]


5. Building the FAISS Vector Database...
FAISS Index built! Total vectors stored: 273518

6. Saving the Database to Disk...
✅ Massive Baseline RAG Backend is ready!
